# Process Addresses Data
1. Ingest the data into the data lakehouse - bronze_addresses 
2. Perform data quality checks and tranform the data as required - silver_addresses_clean
3. Apply canges to the Addresses data (SCD Type 2) - silver_addresses

In [0]:
import dlt
import pyspark.sql.functions as F

##<b> 1. Ingest the data into the data lakehouse - bronze_addresses 
![image_1779128102927.png](./imagens/image_1779128102927.png "image_1779128102927.png")

In [0]:
@dlt.table(
    name = "bronze_addresses",
    table_properties = {'quality' : 'bronze'},
    comment = "Raw addresses data ingested from the source system"
)
def create_bronze_addresses():
  return (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.inferColumnTypes", "true")
    .load("/Volumes/circuitbox/landing/operational_data/addresses/")
    .select(
        "*",
        F.col("_metadata.file_path").alias("input_file_ path"),
        F.current_timestamp().alias("ingest_time") 
    )
  )

## 2. Perform data qaulity checks and transgorm the data as required - **silver_addresses_clean**

In [0]:
@dlt.table(
    name = "silver_addresses_clean",
    comment = "Cleaned addresses data",
    table_properties = {'quality' : 'silver'}
)
@dlt_expect_or_fail("valid_customer_id", "customer_id IS NOT NULL")
@dlt_expect_or_drop("valid_address", "address_line_1 IS NOT NULL")
@dlt.expect("valid_postcode", "LENGTH(postcode) = 5")
def create_silver_addresses_clean():
  return (
    spark.readStream,table("LIVE.bronze_addresses")
        .select(
            "customer_id",
            "address_line_1",
            "city",
            "state",
            "postcode",
            F.col("created_date").cast("date")
        )
)